# Edit-script operations — build, replay, invert

Applies every operation in the inventory to a Latin seed, checks that replaying
the script reproduces the variant, and checks that the inverted script returns
to the seed.

No models and no resources are needed: this exercises the *generation*
direction, where operations are parameterised rather than inferred.


In [1]:
import sys
sys.path.insert(0, ".")

from retexo import (
    CostModel,
    OperationRegistry,
    Scriba,
    VariantBuilder,
)

operation_registry = OperationRegistry.default()
script_verifier = Scriba(operation_registry)
print(f"{len(operation_registry)} operations registered")

18 operations registered


## The operation inventory

In [2]:
operation_registry.to_frame()

,level,role,cost,detectable,requires
tag,,,,,
QUOTE,span,writing,0.00,True,—
FRAME,span,writing,0.50,False,citation_verbs
ADAPT,span,marker,0.00,False,stopwords
DISPERSE,span,marker,1.50,True,clauses
NOP,token,writing,0.00,True,—
MORPH,token,writing,0.25,True,morphology
POS,token,writing,0.50,False,derivation
NE-SUB,token,writing,0.75,False,entities
SYN,token,writing,0.50,True,wordnet


## Example passage

The opening of the *Aeneid*, which the corpus reuses more than any other line.


In [3]:
SEED_PASSAGE = "arma uirumque cano Troiae qui primus ab oris Italiam fato profugus"
seed_passage_tokens = SEED_PASSAGE.split()

for index, token in enumerate(seed_passage_tokens):
    print(f"{index:>2}  {token}")

 0  arma
 1  uirumque
 2  cano
 3  Troiae
 4  qui
 5  primus
 6  ab
 7  oris
 8  Italiam
 9  fato
10  profugus


## Apply operations to the example passage

Each call records one operation. Source positions are named explicitly, reuse
positions are assigned in the order the calls are made.


In [4]:
reuse_builder = VariantBuilder(SEED_PASSAGE, operation_registry)

(
    reuse_builder
    # span: an attribution formula, priced once rather than per token
    .frame(["ut", "ait", "poeta"], detail="attribution")
    # span: a contiguous verbatim core
    .quote([0, 1, 2], detail="verbatim core, 3 tokens")
    # token: same lemma, different inflection
    .inflect(3, "Troia", detail="gen -> abl")
    # token: attested synonym
    .syn(4, "quisquis", detail="wn:r")
    # token: distributional near-synonym
    .syn_dist(5, "primarius", detail="cos=0.71")
    # token: generalization, and its inverse specification
    .hyper(6, "praepositio", detail="ab -> praepositio")
    .hypo(7, "litoribus", detail="oris -> litoribus")
    # cardinality: one token becomes two, then two become one
    .split(8, ["in", "Italiam"], detail="periphrasis")
    .merge([9, 10], "fatoprofugus", detail="compound")
    # structural: a token with no source
    .insert("tandem", detail="connective")
    # markers: counted, but writing nothing
    .mark_reordered(2, detail="cano crosses")
    .mark_adapted([4, 5, 6, 7], detail="substitutions concentrated here")
    .mark_dispersed([9, 10], detail="straddles a clause break")
)

reuse_passage, edit_script = reuse_builder.build()
print("seed passage  :", SEED_PASSAGE)
print("reuse passage :", " ".join(reuse_passage))

seed passage  : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus
reuse passage : ut ait poeta arma uirumque cano Troia quisquis primarius praepositio litoribus in Italiam fatoprofugus tandem


In [5]:
print(edit_script.human())

FRAME                            — -> ut ait poeta              [attribution]
QUOTE           arma uirumque cano -> arma uirumque cano        [verbatim core, 3 tokens]
MORPH                       Troiae -> Troia                     [gen -> abl]
SYN                            qui -> quisquis                  [wn:r]
SYN-DIST                    primus -> primarius                 [cos=0.71]
HYPER                           ab -> praepositio               [ab -> praepositio]
HYPO                          oris -> litoribus                 [oris -> litoribus]
SPLIT                      Italiam -> in Italiam                [periphrasis]
MERGE                fato profugus -> fatoprofugus              [compound]
INS                              — -> tandem                    [connective]
REORDER                       cano -> —                         [cano crosses]
ADAPT           qui primus ab oris -> —                         [substitutions concentrated here]
DISPERSE             fato profugus

In [8]:
print(edit_script.serialize())

FRAME ->0,1,2 ut ait poeta | QUOTE 0,1,2>3,4,5 arma uirumque cano | MORPH 3>6 Troia | SYN 4>7 quisquis | SYN-DIST 5>8 primarius | HYPER 6>9 praepositio | HYPO 7>10 litoribus | SPLIT 8>11,12 in Italiam | MERGE 9,10>13 fatoprofugus | INS ->14 tandem | REORDER 2>- | ADAPT 4,5,6,7>- | DISPERSE 9,10>-


## Apply the remaining operations to another passage

Five operations are left, and the first passage cannot host them: its source
positions are all spent, and `ANT`, `NE-SUB` and `POS` need words that actually
have an antonym, are named entities, or have a derivational partner. `NOP` and
`DEL` are simply easier to read on an uncluttered script.

Generation is parameterised, so operations with no detector — `NE-SUB` and
`POS` here — are still perfectly usable in this direction.


In [6]:
SECOND_SEED_PASSAGE = "Troia tuta manet ubi cognoscat ciuis fortis"

second_reuse_builder = VariantBuilder(SECOND_SEED_PASSAGE, operation_registry)

(
    second_reuse_builder
    .ne_sub(0, "Roma", detail="Troia -> Roma")       # NE-SUB
    .ant(1, "infesta", detail="tuta -> infesta")     # ANT
    .delete(2, detail="dropped from the reuse")      # DEL
    .keep(3, detail="unchanged")                     # NOP
    .pos(4, "cognitio", detail="verb -> noun")       # POS
    .keep(5)
    .keep(6)
)

second_reuse_passage, second_edit_script = second_reuse_builder.build()
print("seed passage  :", SECOND_SEED_PASSAGE)
print("reuse passage :", " ".join(second_reuse_passage))
print()
print(second_edit_script.human())

seed passage  : Troia tuta manet ubi cognoscat ciuis fortis
reuse passage : Roma infesta ubi cognitio ciuis fortis

NE-SUB                       Troia -> Roma                      [Troia -> Roma]
ANT                           tuta -> infesta                   [tuta -> infesta]
DEL                          manet -> —                         [dropped from the reuse]
NOP                            ubi -> ubi                       [unchanged]
POS                      cognoscat -> cognitio                  [verb -> noun]
NOP                          ciuis -> ciuis                   
NOP                         fortis -> fortis                  

cost = 3.25   normalized = 0.464
operations = {'NE-SUB': 1, 'ANT': 1, 'DEL': 1, 'NOP': 3, 'POS': 1}


## Replay the script onto the passage

Scriba executes the script against the seed. It shares no alignment code with
the builder, so agreement is a real check rather than a restatement.


In [ ]:
# Verify the structural validity of the edit script
structural_validity = script_verifier.validate(edit_script)
print("structurally valid :", bool(structural_validity))
if not structural_validity:
    for problem in structural_validity.problems:
        print("  -", problem)

# Replay the edit script on the seed passage and compare with the reuse passage
replayed_tokens = script_verifier.execute(edit_script, seed_passage_tokens)
print("seed passage       :", " ".join(seed_passage_tokens))
print("reuse passage      :", " ".join(reuse_passage))
print("replayed           :", " ".join(replayed_tokens))
print("matches reuse      :", script_verifier.verify(
    edit_script, seed_passage_tokens, reuse_passage))

structurally valid : True
seed passage       : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus
reuse passage      : ut ait poeta arma uirumque cano Troia quisquis primarius praepositio litoribus in Italiam fatoprofugus tandem
replayed           : ut ait poeta arma uirumque cano Troia quisquis primarius praepositio litoribus in Italiam fatoprofugus tandem
matches reuse      : True


## Invert the script to recover the passage

Every operation records both what it consumed and what it wrote, so the script
can be run backwards. Tags map to their inverses: `INS` to `DEL`, `SPLIT` to
`MERGE`, `HYPER` to `HYPO`, and the rest to themselves.


In [10]:
inverse_script = edit_script.invert()
print(inverse_script.human())

FRAME                 ut ait poeta -> —                         [attribution]
QUOTE           arma uirumque cano -> arma uirumque cano        [verbatim core, 3 tokens]
MORPH                        Troia -> Troiae                    [gen -> abl]
SYN                       quisquis -> qui                       [wn:r]
SYN-DIST                 primarius -> primus                    [cos=0.71]
HYPO                   praepositio -> ab                        [ab -> praepositio]
HYPER                    litoribus -> oris                      [oris -> litoribus]
MERGE                   in Italiam -> Italiam                   [periphrasis]
SPLIT                 fatoprofugus -> fato profugus             [compound]
DEL                         tandem -> —                         [connective]
REORDER                          — -> cano                      [cano crosses]
ADAPT                            — -> qui primus ab oris        [substitutions concentrated here]
DISPERSE                         —

In [ ]:
# Replay the inverse edit script on the reuse passage and compare with the seed passage
restored_tokens = script_verifier.execute(inverse_script, reuse_passage)
print("restored     :", " ".join(restored_tokens))
print("seed passage :", SEED_PASSAGE)
print()
print("returns to the seed :", script_verifier.verify(
    inverse_script, reuse_passage, seed_passage_tokens))
print("also for seed two   :", script_verifier.verify(
    second_edit_script.invert(), second_reuse_passage, SECOND_SEED_PASSAGE.split()))

restored     : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus
seed passage : arma uirumque cano Troiae qui primus ab oris Italiam fato profugus

returns to the seed : True
also for seed two   : True
